# 05 — Loss Ratios y Ratio Combinado

## Pregunta de Negocio

¿Es rentable el portafolio de seguros? ¿Cuales lineas de negocio son sustentables?

Los KPIs clave:
- **Loss Ratio** = Perdidas Incurridas / Prima Ganada (< 100% = ganancia tecnica)
- **Expense Ratio** = Gastos / Prima Ganada (tipicamente 25-35%)
- **Combined Ratio** = Loss Ratio + Expense Ratio (< 100% = ganancia de suscripcion)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

LOB_COLORS = {
    'Private Passenger Auto': '#1E3A5F', 'Commercial Auto': '#4A7FA8',
    'Workers Compensation': '#2B6B4F', 'Medical Malpractice': '#8B2E3B',
    'Other Liability': '#C4841D', 'Product Liability': '#5B4A8A',
}
EXPENSE_RATIO = 0.30

ibnr = pd.read_parquet('../data/processed/ibnr_results.parquet')
lob_summary = pd.read_parquet('../data/processed/lob_summary.parquet')
triangles = pd.read_parquet('../data/processed/triangles.parquet')

## 1. Loss Ratio por LOB y Ano

Comparamos la loss ratio reportada (basada en incurrido actual) vs. la loss ratio ultimate (incluyendo IBNR).

In [ ]:
# Compute loss ratios from IBNR results
lr = ibnr.copy()
lr['lr_reported'] = np.where(
    lr['earned_premium'] > 0,
    lr['reported_incurred'] / lr['earned_premium'] * 100,
    0
)
lr['lr_ultimate_paid'] = np.where(
    lr['earned_premium'] > 0,
    lr['ultimate_cl_paid'] / lr['earned_premium'] * 100,
    0
)

# Plot by LOB
fig = make_subplots(rows=2, cols=3, subplot_titles=list(LOB_COLORS.keys()))

for i, (lob, color) in enumerate(LOB_COLORS.items()):
    row = i // 3 + 1
    col = i % 3 + 1
    lob_data = lr[lr['line_of_business'] == lob]
    fig.add_trace(go.Bar(x=lob_data['accident_year'], y=lob_data['lr_reported'],
                        name='Reportada', marker_color=color, showlegend=(i==0)), row=row, col=col)
    fig.add_trace(go.Scatter(x=lob_data['accident_year'], y=lob_data['lr_ultimate_paid'],
                            name='Ultimate (CL)', line=dict(color='#C4841D', dash='dot'),
                            showlegend=(i==0)), row=row, col=col)
    fig.add_hline(y=100, line_dash='dash', line_color='#8B2E3B', row=row, col=col)

fig.update_layout(template='plotly_white', height=600, font_family='Georgia',
                  title='Loss Ratio por LOB (Reportada vs Ultimate)')
fig.show()

## 2. Loss Ratio Agregada por LOB

In [ ]:
lr_agg = lr.groupby('line_of_business').agg(
    total_incurred=('reported_incurred', 'sum'),
    total_paid=('reported_paid', 'sum'),
    total_ultimate=('ultimate_cl_paid', 'sum'),
    total_premium=('earned_premium', 'sum'),
).reset_index()

lr_agg['lr_reported'] = (lr_agg['total_incurred'] / lr_agg['total_premium'] * 100).round(1)
lr_agg['lr_paid'] = (lr_agg['total_paid'] / lr_agg['total_premium'] * 100).round(1)
lr_agg['lr_ultimate'] = (lr_agg['total_ultimate'] / lr_agg['total_premium'] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Bar(x=lr_agg['line_of_business'], y=lr_agg['lr_reported'],
                     name='LR Reportada (Incurrida)', marker_color='#1E3A5F'))
fig.add_trace(go.Bar(x=lr_agg['line_of_business'], y=lr_agg['lr_paid'],
                     name='LR Pagada', marker_color='#4A7FA8'))
fig.add_trace(go.Bar(x=lr_agg['line_of_business'], y=lr_agg['lr_ultimate'],
                     name='LR Ultimate (CL Paid)', marker_color='#C4841D'))
fig.add_hline(y=100, line_dash='dash', line_color='#8B2E3B', annotation_text='Breakeven (100%)')

fig.update_layout(
    template='plotly_white', height=400, font_family='Georgia',
    title='Loss Ratio por LOB — Tres Perspectivas',
    barmode='group', yaxis_title='Loss Ratio (%)',
)
fig.show()

## 3. Combined Ratio

Combined Ratio = Loss Ratio + Expense Ratio. Asumimos un expense ratio de 30% (promedio de la industria).

In [ ]:
# Combined ratio by year (portfolio-wide)
portfolio = lr.groupby('accident_year').agg(
    total_incurred=('reported_incurred', 'sum'),
    total_premium=('earned_premium', 'sum'),
).reset_index()
portfolio['loss_ratio'] = portfolio['total_incurred'] / portfolio['total_premium'] * 100
portfolio['expense_ratio'] = EXPENSE_RATIO * 100
portfolio['combined_ratio'] = portfolio['loss_ratio'] + portfolio['expense_ratio']

fig = go.Figure()
fig.add_trace(go.Bar(x=portfolio['accident_year'], y=portfolio['loss_ratio'],
                     name='Loss Ratio', marker_color='#1E3A5F'))
fig.add_trace(go.Bar(x=portfolio['accident_year'], y=portfolio['expense_ratio'],
                     name='Expense Ratio (30%)', marker_color='#9AB0C8'))
fig.add_trace(go.Scatter(x=portfolio['accident_year'], y=portfolio['combined_ratio'],
                        name='Combined Ratio', line=dict(color='#8B2E3B', width=2),
                        mode='lines+markers'))
fig.add_hline(y=100, line_dash='dash', line_color='#2B6B4F', annotation_text='Breakeven')

fig.update_layout(
    template='plotly_white', height=400, font_family='Georgia',
    title='Combined Ratio por Ano de Accidente',
    barmode='stack', yaxis_title='Ratio (%)', xaxis_title='Ano de Accidente',
)
fig.show()

## 4. Waterfall: De Prima a Resultado

In [ ]:
# Portfolio-level waterfall
total_premium = lr['earned_premium'].sum()
total_losses = lr['reported_incurred'].sum()
total_expenses = total_premium * EXPENSE_RATIO
underwriting_result = total_premium - total_losses - total_expenses

fig = go.Figure(go.Waterfall(
    x=['Prima Ganada', 'Perdidas Incurridas', 'Gastos (30%)', 'Resultado Tecnico'],
    y=[total_premium, -total_losses, -total_expenses, underwriting_result],
    measure=['absolute', 'relative', 'relative', 'total'],
    text=[f'${v/1e6:.1f}M' for v in [total_premium, -total_losses, -total_expenses, underwriting_result]],
    textposition='outside',
    connector={'line': {'color': '#E5E4DF'}},
    increasing={'marker': {'color': '#2B6B4F'}},
    decreasing={'marker': {'color': '#8B2E3B'}},
    totals={'marker': {'color': '#1E3A5F'}},
))
fig.update_layout(
    template='plotly_white', height=400, font_family='Georgia',
    title='Waterfall: De Prima Ganada a Resultado de Suscripcion',
)
fig.show()

In [ ]:
# Summary table
lr_agg['expense_ratio'] = EXPENSE_RATIO * 100
lr_agg['combined_ratio'] = lr_agg['lr_reported'] + lr_agg['expense_ratio']
lr_agg['profitable'] = lr_agg['combined_ratio'] < 100

lr_agg[['line_of_business', 'lr_reported', 'lr_paid', 'lr_ultimate',
         'expense_ratio', 'combined_ratio', 'profitable']].sort_values('combined_ratio')

## So What?

- **Private Passenger Auto** y **Product Liability** son las unicas lineas con combined ratio < 100% — rentables en suscripcion.
- **Medical Malpractice** (combined ~310%) y **Workers' Comp** (~259%) muestran perdidas severas. En la practica, estas lineas sobreviven por ingreso de inversiones y reaseguro.
- La brecha entre loss ratio reportada y ultimate (CL paid) indica cuanto IBNR adicional se necesita — mayor en lineas de cola larga.
- Un expense ratio de 30% es una aproximacion conservadora; en la practica varia por linea y compania (20-40%).
- **Recomendacion**: Las companias deberian evaluar la viabilidad de med mal y workers' comp, o asegurar precios adecuados y controles de riesgo mas estrictos.